# NB-03 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-03**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-03.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This teaching extension makes the volume’s commitment and intertemporal pricing ideas executable. Its small approved fixture illustrates the mechanisms; it is not an ISO implementation.

- [Volume I, §12 — Unit commitment, non-convexity, and uplift](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s12)
- [Volume I, §13 — Ramp constraints and multi-interval pricing](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s13)
- [Volume I, §11 — Scarcity pricing and co-optimization](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s11)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-03', require_key=True)
print(setup_status)


# NB-03 — Commitment and time-coupled pricing
Companion to Volume I's commitment, reserves and ramping discussion.
Prerequisites: NB-01 and linear-program dual prices. Learn to separate binary
commitment from continuous dispatch, then account for intertemporal scarcity.
We first add reserve to an energy-only problem, then add ramps, compare myopic
dispatch, and reconcile cost recovery.

**User-approved new deterministic teaching fixture:** two generators and four
one-hour intervals. No random market data or actual ISO settlement rules.

In [ ]:
NOTEBOOK_ID='NB-03'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

cfg=inputs['config']
print('Original approved teaching configuration:',json.dumps(cfg,indent=2))
def dispatch(name,request,parents=()):
    value=execute(name,'commitment_ramp',request,parents)['intertemporal_dispatch']
    if value['feasibility_status']=='infeasible':
        assert not value['prices_available'] and not value['dispatch_available']
        assert 'energy_prices' not in value and 'generation_mw' not in value
        return value
    assert value['feasibility_status']=='feasible' and not value['shortage_slack_used']
    # Rename fields for the original plots; no local dispatch or price calculation.
    return dict(value,generation=np.asarray(value['generation_mw']),reserve=np.asarray(value['reserve_mw']),
        on=np.asarray(value['commitment']),energy_prices=np.asarray(value['energy_prices']),
        reserve_prices=np.asarray(value['reserve_prices']))
def toy_ledger(value):
    item=value['toy_make_whole_ledger']
    assert item['policy']==inputs['accounting_policy'] and not item['market_rule_certified']
    return {k:np.asarray(item[k]) for k in ('cost','energy_and_reserve_revenue','toy_make_whole','profit_before_uplift')}

## Binary schedules outside, continuous dispatch inside
For each of the 256 binary schedules u, solve an LP in energy g and reserve r:
minimize sum(c g + no-load u + startup max(u−u_previous,0)). Balance g₁+g₂=D
in every interval; enforce minimum u ≤ g and g+r ≤ capacity u.
Reserve sums to at least 20 MW; only generator 1 is eligible. This eligibility
is a declared teaching assumption. Reserve has zero offer cost and represents
headroom; deployment trajectories and sustained delivery are not modeled.

Ramps constrain both upward and downward changes, including the initial state
and on/off transitions. Minimum up/down times, network constraints, losses,
startup ramp allowances and terminal obligations are omitted. Starts incur
costs but there are no shutdown costs. Initial generation is [80,0] MW.
Enumeration selects the least-cost feasible integer schedule. **Only after
fixing that schedule** do the continuous LP duals have a marginal-price meaning.

In [ ]:
cases={name:dispatch(name,request) for name,request in inputs['requests'].items()}
assert all(v['schedules_considered']==256 for v in cases.values())
for name,r in cases.items():
    np.testing.assert_allclose(r['generation'].sum(axis=1),cfg['demand_mw'])
    print(name,'cost $',r['total_cost'],'feasible schedules',r['feasible_schedules'],'on/off',r['on'].tolist())
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,(name,r) in zip(axes,cases.items()):
    ax.stackplot(range(4),r['generation'].T,labels=['Generator 1','Generator 2'],colors=['#88C0D0','#B48EAD']);ax.plot(range(4),cfg['demand_mw'],'o--',color='#ECEFF4',label='Demand');ax.set(title=name,xlabel='Interval (one hour)',ylabel='MW',xticks=range(4),ylim=(0,230))
axes[0].legend(fontsize=8);fig.tight_layout();plt.show()

## Look ahead before the peak
The energy-only solution starts the expensive generator just for the peak.
Reserve reduces cheap available energy. Ramps then require the expensive unit
to start in interval 0 and produce 20,50,80,50 MW. Cheap generation falls early
to make room for that necessary preparation. The sequential optimizer commits
to each cheapest hourly decision and fails at interval 2; it is not a feasible
lower-cost alternative to the horizon plan. No shortage slack hides the failure.

In [ ]:
rows=[];failed=None;previous=None
for t in range(4):
    request=copy.deepcopy(inputs['requests']['With ramps'])
    request['inputs']['demand_panel']=dict(period_ids=[str(t)],demand_mw=[cfg['demand_mw'][t]])
    limits=request['inputs']['intertemporal_limits']
    limits.update(duration_hours=[1.],reserve_mw=[cfg['reserve_mw'][t]])
    if rows:
        limits.update(initial_generation_mw=rows[-1]['generation'][-1].tolist(),initial_on=rows[-1]['on'][-1].tolist())
    name='myopic:'+str(t)
    changed=dispatch(name,request,[previous] if previous else [])
    if changed['feasibility_status']=='infeasible':
        failed=t
        break
    rows.append(changed);previous=name
print('Myopic completed generation:',[item['generation'].tolist() for item in rows]);print('First infeasible interval:',failed)
r=cases['With ramps'];assert failed==2
fig,axes=plt.subplots(1,2,figsize=(11,4))
for (name,q),color in zip(cases.items(),['#88C0D0','#B48EAD','#A3BE8C']):
    axes[0].plot(range(4),q['energy_prices'],marker='o',label=name,color=color)
axes[0].set(xlabel='Interval',ylabel='Energy dual ($/MWh)',xticks=range(4),title='Fixed-commitment energy prices');axes[0].legend(fontsize=8)
axes[1].plot(range(4),r['reserve_prices'],'o-',color='#EBCB8B');axes[1].set(xlabel='Interval',ylabel='Reserve dual ($/MW for one hour)',xticks=range(4),title='Opportunity cost of eligible headroom');fig.tight_layout();plt.show()

## Boundary sensitivities are directional
The peak energy dual is $200/MWh, above either fuel offer: earlier ramping
decisions also change when peak demand changes. This schedule sits on a
feasibility boundary. A small demand decrease in interval 1 or increase in
interval 2 is infeasible with commitment fixed. Report those directions as
infeasible, not as finite prices. Feasible one-sided differences below agree
with the selected dual; uniqueness and two-sided differentiability are not
claimed. These are not derivatives of the integer optimum or actual market LMPs.

In [ ]:
checks=[];directional_results=[]
for t in range(4):
    sides=[]
    for sign in [-1,1]:
        perturbation=next(p for p in inputs['perturbations'] if p['period']==t and p['sign']==sign)
        request=copy.deepcopy(inputs['requests']['With ramps'])
        request['inputs']['demand_panel']['demand_mw']=perturbation['demand_mw']
        request['inputs']['intertemporal_limits'].update(boundary='single_bus_fixed_commitment',commitment=r['on'].tolist())
        request['parameters']=dict(variant='ramp',execution_profile='explicit_feasibility',accounting='toy_make_whole')
        changed=dispatch(f'directional:{t}:{sign}',request,['With ramps'])
        slope=None if changed['feasibility_status']=='infeasible' else (changed['total_cost']-r['total_cost'])/(sign*perturbation['step_mw'])
        if slope is not None:assert np.isclose(slope,r['energy_prices'][t],atol=1e-5)
        sides.append(slope);directional_results.append(dict(period=t,sign=sign,slope=slope,result=changed))
    checks.append(sides)
    print('Interval',t,'down/up slopes',sides,'selected dual',r['energy_prices'][t])

## Uplift is a separate accounting choice
Energy plus reserve revenue may fail to cover startup and no-load costs.
Our toy make-whole is max(total cost − energy/reserve revenue,0), computed per
generator across the full horizon. It omits eligibility, lost opportunity,
contract revenues and all jurisdiction-specific rules. Never add startup cost
directly to an LP dual. The energy-only case needs $630 for generator 2;
the reserve case needs $1,860. The ramp case needs none under these selected
duals. Scarcity revenue can cover fixed costs but need not always do so.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
for i,(name,q) in enumerate(cases.items()):
    ledger=toy_ledger(q);print(name,{k:v.tolist() for k,v in ledger.items()})
    ax.bar(i,ledger['toy_make_whole'].sum(),color=['#88C0D0','#B48EAD','#EBCB8B'][i])
ax.set(xticks=range(3),xticklabels=list(cases),ylabel='Toy make-whole ($)',title='Fixed-cost recovery is distinct from marginal pricing');plt.show()

## Exercise: what disappears without ramps?
Compare peak energy duals in the reserve-only and ramp-constrained cases.
Which cost is missed by treating hours independently? Use the already solved
cases; no new fixture is needed. The answer below is an opportunity-cost
comparison, not a universal decomposition valid for arbitrary commitments.

In [ ]:
print('Peak dual increase from time coupling:',cases['With ramps']['energy_prices'][2]-cases['With reserve']['energy_prices'][2])
assert np.isclose(toy_ledger(cases['Energy only'])['toy_make_whole'][1],630)

In [ ]:
bad=copy.deepcopy(inputs['requests']['With ramps']);bad['parameters']['execution_profile']='unsupported';reject('commitment_ramp',bad)
bad=copy.deepcopy(inputs['requests']['With ramps']);bad['inputs']['intertemporal_offers']['reserve_eligible'].pop();reject('commitment_ramp',bad)
def plain(value):
    if isinstance(value,np.ndarray):return value.tolist()
    raise TypeError(type(value))
summary_data=dict(cases=cases,ledgers={k:toy_ledger(v) for k,v in cases.items()},myopic=dict(completed=rows,first_infeasible_interval=failed),directional_results=directional_results,directional_energy_checks=checks)
(session.output/'consumer-summary.json').write_text(json.dumps(summary_data,default=plain,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(all_256_schedules_evaluated=True,myopic_infeasibility_preserved=True,eight_directional_checks=True,
    conditional_prices_only=True,toy_accounting_not_market_uplift=True,original_approved_fixture=True,new_synthetic_data=False))
print(summary)